# AeroPulse Anomaly — Phase 1: Import Canonical Dataset

Import the persisted pm25_estimator dataset. Do **not** re-fetch OpenAQ, Open-Meteo, or FIRMS.


## Objective

Create a **canonical anomaly-detector base table once**, by importing the pm25 Phase-1/Phase-2 Parquet already sitting on disk.

This notebook deliberately does **not**:

- call OpenAQ / FIRMS / Open-Meteo
- fit an hour-of-week baseline (that happens on train rows only, in Phase 3)
- train a model

Prefer the event-aware table (lags, fire rolls, stagnation). Fall back to `base_dataset.parquet` if Phase 2 has not been run.


## Target dataset contract

Every row must have `timestamp_utc`, `location_id`, `lat`, `lon`, `pm25`, and the Open-Meteo weather columns (`temperature_2m`, `relative_humidity_2m`, `surface_pressure`, `precipitation`, `wind_speed_10m`, `wind_direction_10m`, `boundary_layer_height`). Fire and lag columns are kept when present.


In [1]:
# This project's .venv is managed by uv and has NO pip, so `%pip install` fails here.
# Install from the shell at the repo root:
#
#     uv pip install --python .venv/bin/python -r AeroPulse_ML_Notebooks/requirements.txt
#
# Then verify the kernel prints .../AeroPulse/.venv/bin/python
import sys
print("kernel:", sys.executable)


kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
# ============================================================================
# CONFIGURATION
# Load .env (this folder, or a parent — find_dotenv walks upward).
# Paths are folder-local so each model writes its own Parquet/artifacts.
# ============================================================================
from pathlib import Path
import os, json, math, warnings, hashlib, time
import numpy as np
import pandas as pd

from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
if not _dotenv_path:
    raise RuntimeError(
        "No .env found. Copy this folder's .env.example to .env, "
        "or launch Jupyter from a directory whose parent has .env "
        "(OPENAQ_API_KEY is only required for optional network steps)."
    )
load_dotenv(_dotenv_path, override=True)
print("Loaded env from:", _dotenv_path)

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(os.getenv("AEROPULSE_PROJECT_ROOT", ".")).resolve()
PM25_DATA_ROOT = Path(os.getenv("PM25_DATA_ROOT", "../pm25_estimator")).resolve()
PM25_PROCESSED = PM25_DATA_ROOT / "data" / "pm25" / "processed"

DATA_ROOT = PROJECT_ROOT / "data" / "anomaly"
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
BASE_DIR = PROCESSED_DIR / "base"
FEATURE_DIR = PROCESSED_DIR / "features"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "anomaly"
PHASE3_DIR = ARTIFACT_DIR / "phase3"
for p in [RAW_DIR, PROCESSED_DIR, BASE_DIR, FEATURE_DIR, ARTIFACT_DIR, PHASE3_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def dataset_fingerprint(frame: pd.DataFrame) -> str:
    payload = {
        "rows": int(len(frame)),
        "cols": list(frame.columns),
        "tmin": str(frame["timestamp_utc"].min()) if "timestamp_utc" in frame.columns else None,
        "tmax": str(frame["timestamp_utc"].max()) if "timestamp_utc" in frame.columns else None,
        "pm25_sum": float(pd.to_numeric(frame["pm25"], errors="coerce").sum()) if "pm25" in frame.columns else None,
    }
    return hashlib.sha256(json.dumps(payload, default=str).encode()).hexdigest()

def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}. Set it in .env.")
    return value

CONFIG = {
    "pm25_data_root": str(PM25_DATA_ROOT),
    "schema_version": "1.0.0",
    "very_poor_ugm3": float(os.getenv("ANOMALY_VERY_POOR_UGM3", "121")),
}
print("Project:", PROJECT_ROOT)
print("PM25 data:", PM25_DATA_ROOT)
print("Data root:", DATA_ROOT)
print("Configuration:", CONFIG)


Loaded env from: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/.env
Project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector
PM25 data: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
Data root: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/data/anomaly
Configuration: {'pm25_data_root': '/Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator', 'schema_version': '1.0.0', 'very_poor_ugm3': 121.0}


## 1. Locate pm25 Parquet

Expected files (from `PM25_DATA_ROOT`):

```text
data/pm25/processed/event_aware/pm25_event_aware_features.parquet   # preferred
data/pm25/processed/base/base_dataset.parquet                      # fallback
```


In [3]:
# ============================================================================
# LOAD UPSTREAM PM2.5 DATASET
# Event-aware is preferred because Phase 2 already built leakage-safe lags,
# fire 24h aggregates, and stagnation flags. Baseline fitting is still deferred.
# ============================================================================
EVENT_FILE = PM25_PROCESSED / "event_aware" / "pm25_event_aware_features.parquet"
BASE_FILE = PM25_PROCESSED / "base" / "base_dataset.parquet"

if EVENT_FILE.exists():
    src = EVENT_FILE
    source_name = "pm25_phase2_event_aware"
elif BASE_FILE.exists():
    src = BASE_FILE
    source_name = "pm25_phase1_base"
else:
    raise FileNotFoundError(
        f"Neither {EVENT_FILE} nor {BASE_FILE} exists. "
        "Run pm25_estimator Phase 1 (and ideally Phase 2) first."
    )

print("Importing:", src)
df = pd.read_parquet(src)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

REQUIRED = [
    "timestamp_utc", "location_id", "lat", "lon", "pm25",
    "temperature_2m", "relative_humidity_2m", "surface_pressure",
    "precipitation", "wind_speed_10m", "wind_direction_10m",
    "boundary_layer_height",
]
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise ValueError(f"Upstream dataset is missing required columns: {missing}")

print("source:", source_name)
print("rows:", len(df), "stations:", df["location_id"].nunique())
print("window:", df["timestamp_utc"].min(), "→", df["timestamp_utc"].max())
print("columns:", len(df.columns))


Importing: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/processed/event_aware/pm25_event_aware_features.parquet


source: pm25_phase2_event_aware
rows: 1705252 stations: 149
window: 2025-02-18 20:30:00+00:00 → 2026-09-09 07:30:00+00:00
columns: 208


### Result — imported dataset

Expect ~1.7M rows / 149 stations / Feb 2025–Sep 2026 if you imported the current India canonical table. If the row count is tiny, you pointed `PM25_DATA_ROOT` at the wrong folder.


## 2. Persist the anomaly base table

Raw provider data stays in pm25_estimator. This folder gets its own copy so later phases never depend on another notebook's working directory.


In [4]:
# ============================================================================
# PERSIST + MANIFEST
# ============================================================================
out = BASE_DIR / "anomaly_base.parquet"
df.to_parquet(out, index=False)

manifest = {
    "schema_version": CONFIG["schema_version"],
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "source": source_name,
    "source_path": str(src),
    "rows": int(len(df)),
    "stations": int(df["location_id"].nunique()),
    "columns": list(df.columns),
    "window_start_utc": str(df["timestamp_utc"].min()),
    "window_end_utc": str(df["timestamp_utc"].max()),
    "fingerprint": dataset_fingerprint(df[["timestamp_utc", "location_id", "pm25"]]),
    "notes": [
        "Imported from pm25_estimator. No API calls.",
        "Baseline / residual are NOT computed here — they leak if fit on the full table.",
    ],
}
with open(BASE_DIR / "dataset_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print("PHASE 1 COMPLETE")
print(" wrote", out)
print(" wrote", BASE_DIR / "dataset_manifest.json")
print("Next: 02_anomaly_phase2_residual_features.ipynb")


PHASE 1 COMPLETE
 wrote /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/data/anomaly/processed/base/anomaly_base.parquet
 wrote /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/data/anomaly/processed/base/dataset_manifest.json
Next: 02_anomaly_phase2_residual_features.ipynb
